# Argumentation Mining em Community Notes BR — pipeline final (v2)

**Versão:** v2 — pipeline reorganizado em etapas logicamente encadeadas, com correções aplicadas após a primeira entrega:

- `noteId` preservado como string em todo o pipeline (evita perda de precisão do `pd.read_json` para inteiros > 2⁵³).
- Classificador de meta-nota (`is_meta`) aplicado **antes** da seleção qualitativa, com filtro embutido.
- `try/finally` em torno de `ENABLE_THINKING` na recaptura de reasoning.
- Bloco novo de tradução PT-BR dos reasoning (preserva originais).
- Consolidação final unificada (`dataset_anotado_final.parquet`) sem duplicação entre células.

**Objetivo:** comparar duas estratégias de extração de spans argumentativos ({CLAIM, EVIDENCIA, FONTE, QUALIFICADOR}) sobre notas de checagem da Community Notes BR — E1 (regras léxico-sintáticas + spaCy local) × E2 (LLM Qwen3.6-max-preview via DashScope, com protocolo snippet→offset).

## 1. Setup

In [ ]:
# Descomente na primeira execução
!pip install -q huggingface_hub pandas pyarrow tqdm spacy openai gdown scikit-learn
!python -m spacy download pt_core_news_md

In [ ]:
from __future__ import annotations

import json
import os
import re
import time
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Iterable, Literal

import pandas as pd
from tqdm.auto import tqdm

RANDOM_SEED = 42

# Universo (HF)
HF_REPO = 'histlearn/notas-comunidade-ptbr'
NOTES_FILE = 'notes_pt.parquet'
ENTITIES_FILE = 'entities.parquet'
# Revisao PINADA do dataset (o upstream evolui: entities.parquet foi pos-processado
# de 536.373 para 510.352 linhas em jun/2026 — ver CHANGELOG.md do dataset).
HF_DATASET_REVISION = 'ce366492f8a9a9a5fbe0b17b223cb19ae2c54535'

# Tweets hidratados (Drive)
TWEETS_DRIVE_FILE_ID = '1EwmIQeTGIEpfDUNTy7Ev0yai1TfWn09i'

# Texto anotável da nota: `summary` (preserva URLs como FONTE).
TEXT_FIELD = 'summary'
FALLBACK_TEXT_FIELD = 'summary_clean'

# Quotas do subset tweet-aware (ver §3).
QUOTA_NMR = 1500           # amostrado de ~3377
QUOTA_NMR_CAP_PER_TWEET = 5  # evita dominância de tweets virais
MIN_TEXT_LEN = 20

# Paths
DATA_DIR = Path('data')
RUNS_DIR = DATA_DIR / 'runs'
for d in (DATA_DIR, RUNS_DIR):
    d.mkdir(parents=True, exist_ok=True)

SUBSET_PATH = DATA_DIR / 'subset_final.parquet'
TWEETS_PATH = DATA_DIR / 'tweets_hidratados.parquet'
SWEEP_JSONL = RUNS_DIR / 'sweep_final.jsonl'
AGREEMENT_PARQUET = RUNS_DIR / 'agreement_final.parquet'
TWEET_ALIGN_PARQUET = RUNS_DIR / 'tweet_alignment_final.parquet'
QUALITATIVE_PARQUET = RUNS_DIR / 'qualitative_60_final.parquet'
QUALITATIVE_XLSX = RUNS_DIR / 'qualitative_60_final.xlsx'
REASONING_JSONL = RUNS_DIR / 'qualitative_60_reasoning.jsonl'

print('paths configurados em', DATA_DIR.resolve())

In [ ]:
# Registro do ambiente (reprodutibilidade)
import platform
import importlib.metadata as _md

def _v(pkg):
    try:
        return _md.version(pkg)
    except _md.PackageNotFoundError:
        return '(ausente)'

print('python', platform.python_version(), '|', platform.platform())
for _p in ['pandas', 'pyarrow', 'spacy', 'openai', 'huggingface_hub',
           'scikit-learn', 'tqdm', 'gdown']:
    print(f'  {_p:<16} {_v(_p)}')

## 2. Aquisição de dados

### 2.1 Tweets hidratados (Drive)

In [ ]:
# Tweets hidratados via gdown
if not TWEETS_PATH.exists():
    import gdown
    url = f'https://drive.google.com/uc?id={TWEETS_DRIVE_FILE_ID}'
    gdown.download(url, str(TWEETS_PATH), quiet=False)
else:
    print('tweets_hidratados.parquet já presente:', TWEETS_PATH)

tweets_raw = pd.read_parquet(TWEETS_PATH)
print(f'tweets brutos: {len(tweets_raw):,}')

tweets = tweets_raw[
    tweets_raw['lang'].eq('pt') &
    (tweets_raw['text'].fillna('').str.len() > 20)
].copy()
tweets['tweet_id'] = tweets['tweet_id'].astype(str)
tweets = tweets[['tweet_id', 'text', 'lang', 'stratum']].rename(columns={'text': 'tweet_text'})
print(f'tweets utilizáveis (pt + texto > 20): {len(tweets):,}')
print('\ndistribuição por stratum (priorização da coleta):')
print(tweets['stratum'].value_counts().to_string())

### 2.2 Notas (Hugging Face)

In [ ]:
# notes_pt.parquet via HF
from huggingface_hub import hf_hub_download

print('baixando notes_pt.parquet...')
notes_path = hf_hub_download(HF_REPO, NOTES_FILE, repo_type='dataset',
                             revision=HF_DATASET_REVISION)
notes_all = pd.read_parquet(notes_path)
notes_all['tweetId'] = notes_all['tweetId'].astype(str)
print(f'notas no universo HF: {len(notes_all):,}')
print('colunas relevantes:', [c for c in notes_all.columns if c in
      ('noteId', 'tweetId', 'summary', 'summary_clean', 'consenso',
       'macrotheme_label', 'classification_algoritmo', 'status_final')])

## 3. Subset tweet-aware

### 3.1 Cruzamento nota × tweet

In [ ]:
# Cruza notas com tweets utilizaveis
tweet_ids = set(tweets['tweet_id'])
covered = notes_all[notes_all['tweetId'].isin(tweet_ids)].copy()
print(f'universo tweet-aware bruto: {len(covered):,} notas em {covered["tweetId"].nunique():,} tweets')
print('\ndistribuição natural por consenso:')
print(covered['consenso'].value_counts(dropna=False).to_string())

### 3.2 Filtro de qualidade mínima

In [ ]:
# Filtro de qualidade minima (texto util >= 20 chars apos colapso de whitespace)
def _pick_text(row):
    txt = row.get(TEXT_FIELD)
    if pd.isna(txt) or str(txt) == '':
        txt = row.get(FALLBACK_TEXT_FIELD)
    return '' if pd.isna(txt) else str(txt)

covered['_text'] = covered.apply(_pick_text, axis=1)
covered['_len_clean'] = covered['_text'].str.replace(r'\s+', ' ', regex=True).str.strip().str.len()
covered = covered[covered['_len_clean'] >= MIN_TEXT_LEN].reset_index(drop=True)
print(f'após filtro de qualidade mínima: {len(covered):,} notas')

### 3.3 Quotas por consenso (cap por tweet em NMR)

In [ ]:
# Quotas por consenso com cap em NMR
def build_subset(df, quota_nmr, cap_nmr, seed):
    parts = []
    # CRH, CRNH, Outro: todos
    for cons in ['CRH', 'CRNH', 'Outro']:
        parts.append(df[df['consenso'].eq(cons)].copy())
    # NMR: cap por tweet primeiro, depois amostra
    nmr = df[df['consenso'].eq('NMR')].copy()
    nmr_capped = nmr.sample(frac=1, random_state=seed).groupby('tweetId').head(cap_nmr)
    if len(nmr_capped) > quota_nmr:
        nmr_capped = nmr_capped.sample(n=quota_nmr, random_state=seed)
    parts.append(nmr_capped)
    return pd.concat(parts).sample(frac=1, random_state=seed).reset_index(drop=True)

subset = build_subset(covered, QUOTA_NMR, QUOTA_NMR_CAP_PER_TWEET, RANDOM_SEED)
# Anexa o tweet_text
subset = subset.merge(tweets[['tweet_id', 'tweet_text', 'stratum']],
                      left_on='tweetId', right_on='tweet_id', how='left')

print(f'subset final: {len(subset):,} notas')
print('\n--- composição por consenso ---')
print(subset['consenso'].value_counts(dropna=False).to_string())
print('\n--- tweets distintos representados ---')
print(f'{subset["tweetId"].nunique():,} tweets')
print('\n--- notas por tweet (após cap em NMR) ---')
print(subset.groupby('tweetId').size().describe(percentiles=[.5, .9, .99]).round(1).to_string())

subset.to_parquet(SUBSET_PATH, index=False)
print(f'\nsalvo em {SUBSET_PATH}')

## 4. Schema de spans (validação + normalização)

In [ ]:
SpanType = Literal['CLAIM', 'EVIDENCIA', 'FONTE', 'QUALIFICADOR']
LABELS: tuple[SpanType, ...] = ('CLAIM', 'EVIDENCIA', 'FONTE', 'QUALIFICADOR')
LABEL_SET = set(LABELS)

@dataclass(frozen=True)
class Span:
    start: int
    end: int
    type: str

    def text(self, source: str) -> str:
        return source[self.start:self.end]

    def to_dict(self) -> dict:
        return asdict(self)

def validate_spans(spans: list[Span], text_len: int) -> list[Span]:
    out = []
    seen = set()
    for s in spans:
        if s.type not in LABEL_SET:
            continue
        if s.start < 0 or s.end > text_len or s.start >= s.end:
            continue
        key = (s.start, s.end, s.type)
        if key in seen:
            continue
        seen.add(key)
        out.append(s)
    return out

# Recorta lixo curto antes de uma URL ("t/ https://..."), mas preserva nome de veiculo
# antes da URL ("Folha de S.Paulo (https://folha.com)").
URL_NOISE_PREFIX_MAX = 3

def _clean_span_boundaries(span: Span, text: str) -> Span:
    start, end = span.start, span.end
    while start < end and text[start].isspace():
        start += 1
    while end > start and text[end - 1].isspace():
        end -= 1
    if span.type == 'FONTE':
        frag = text[start:end]
        url_match = re.search(r'https?://\S+', frag)
        if url_match and url_match.start() <= URL_NOISE_PREFIX_MAX:
            start = start + url_match.start()
            end = start + len(url_match.group(0))
        while end > start and text[end - 1] in '.,;:)]}':
            end -= 1
    return Span(start, end, span.type)

def normalize_spans(spans: list[Span], text: str, overlap_threshold: float = 0.8) -> list[Span]:
    """Remove duplicatas e overlaps quase identicos, preservando spans sobrepostos legitimos."""
    valid = validate_spans([_clean_span_boundaries(s, text) for s in spans], len(text))
    selected: list[Span] = []
    for s in sorted(valid, key=lambda x: (x.type, -(x.end - x.start), x.start)):
        slen = max(1, s.end - s.start)
        redundant = False
        for kept in selected:
            if s.type != kept.type:
                continue
            klen = max(1, kept.end - kept.start)
            ov = max(0, min(s.end, kept.end) - max(s.start, kept.start))
            if ov / max(slen, klen) >= overlap_threshold:
                redundant = True
                break
        if not redundant:
            selected.append(s)
    return sorted(selected, key=lambda x: (x.start, x.end, x.type))

def _find_entities_file() -> Path | None:
    for c in (ENTITIES_PATH, Path('entities.parquet'), Path('..') / 'entities.parquet'):
        if c.exists():
            return c
    return None

SOURCE_ENTITY_TYPES = {'URL_DOMINIO', 'VEICULO_MIDIA', 'FONTE_CITADA', 'ORGAO_PUBLICO'}

def _contained_in_any(span: tuple[int, int], containers: list[tuple[int, int]]) -> bool:
    return any(span[0] >= c[0] and span[1] <= c[1] for c in containers)

def load_entity_source_spans(df: pd.DataFrame) -> dict[str, list[Span]]:
    """Carrega FONTE de entities.parquet RE-LOCALIZANDO cada entidade pela superficie.

    Os offsets `inicio`/`fim` da tabela sao automaticos e frequentemente nao se referem
    ao texto que anotamos: a auditoria publica do proprio dataset
    (`quality_audit/entity_offset_audit.parquet`) marca ~51% deles como fora do texto de
    referencia. Por isso NAO usamos os offsets diretamente: aceitamos o offset apenas
    quando ele reproduz exatamente `texto_entidade` no nosso texto; caso contrario,
    procuramos a superficie no texto e usamos o offset original so como desambiguador
    (ocorrencia mais proxima). Entidades cuja superficie nao ocorre no texto sao
    descartadas e contadas no relatorio.
    """
    from collections import Counter

    found = _find_entities_file()
    if found is None:
        from huggingface_hub import hf_hub_download
        found = Path(hf_hub_download(HF_ENRICHED_REPO, 'entities.parquet',
                                     repo_type='dataset', revision=HF_DATASET_REVISION))
        print(f'entities.parquet baixado do HF (rev {HF_DATASET_REVISION[:8]}): {found}')
    else:
        print(f'entities.parquet carregado de: {found.resolve()}')

    use_ids = set(df['noteId'].astype(str))
    entities = pd.read_parquet(found)
    entities['noteId'] = entities['noteId'].astype(str)
    entities = entities[entities['noteId'].isin(use_ids)]
    entities = entities[entities['papel_no_texto'].eq('fonte_ou_evidencia')]
    entities = entities[entities['tipo_entidade'].isin(SOURCE_ENTITY_TYPES)].copy()

    texts = dict(zip(df['noteId'].astype(str), df['_text']))

    def _relocate(text: str, surface: str, hint: int | None) -> int | None:
        """Ocorrencias exatas de `surface` em `text`; devolve a mais proxima de `hint`."""
        if not surface:
            return None
        hits = []
        pos = text.find(surface)
        while pos >= 0:
            hits.append(pos)
            pos = text.find(surface, pos + 1)
        if not hits:
            return None
        if hint is None:
            return hits[0]
        return min(hits, key=lambda p: abs(p - hint))

    stats = Counter()
    located: dict[str, list[tuple[int, int, str]]] = {nid: [] for nid in use_ids}
    for r in entities.itertuples(index=False):
        note_id = str(r.noteId)
        text = texts.get(note_id, '')
        surface = str(getattr(r, 'texto_entidade', '') or '')
        try:
            hint = int(r.inicio)
        except (TypeError, ValueError):
            hint = None
        if (surface and hint is not None and 0 <= hint < len(text)
                and text[hint:hint + len(surface)] == surface):
            start = hint
            stats['offset_exato'] += 1
        else:
            start = _relocate(text, surface, hint)
            if start is None:
                stats['descartada_nao_encontrada'] += 1
                continue
            stats['relocalizada'] += 1
        located[note_id].append((start, start + len(surface), str(r.tipo_entidade)))

    by_note: dict[str, list[Span]] = {}
    for note_id, items in located.items():
        if not items:
            continue
        url_ranges = [(a, b) for a, b, t in items if t == 'URL_DOMINIO']
        spans = []
        for a, b, t in items:
            if t != 'URL_DOMINIO' and _contained_in_any((a, b), url_ranges):
                stats['descartada_dentro_de_url'] += 1
                continue
            spans.append(Span(a, b, 'FONTE'))
        by_note[note_id] = validate_spans(spans, len(texts.get(note_id, '')))

    total = sum(len(v) for v in by_note.values())
    n_notes = sum(1 for v in by_note.values() if v)
    detalhe = ', '.join(f'{k}={v:,}' for k, v in sorted(stats.items()))
    print(f'entidades FONTE: {total:,} spans em {n_notes:,} notas | {detalhe}')
    return by_note

## 5. FONTE via entidades pré-extraídas (HF)

Os offsets `inicio`/`fim` da tabela de entidades são automáticos e nem sempre se referem ao
texto que anotamos — a auditoria pública do próprio dataset
(`quality_audit/entity_offset_audit.parquet`) marca **~51 %** deles como fora do texto de
referência. O loader da §4, por isso, **re-localiza cada entidade pela superfície**
(`texto_entidade`) no nosso texto: o offset original só é aceito quando reproduz a
superfície exatamente, e serve como desambiguador quando ela se repete. Entidades cuja
superfície não ocorre no texto são descartadas e contadas no relatório impresso.

In [ ]:
ENTITIES_PATH = DATA_DIR / ENTITIES_FILE
HF_ENRICHED_REPO = HF_REPO

ENTITY_SOURCE_SPANS = load_entity_source_spans(subset)

## 6. Parser local — spaCy `pt_core_news_md`

In [ ]:
import spacy

SPACY_MODEL = 'pt_core_news_md'
SPACY_BATCH_SIZE = 64

try:
    nlp = spacy.load(SPACY_MODEL, disable=['ner'])
except OSError as e:
    raise OSError(
        f'modelo spaCy {SPACY_MODEL} ausente; rode `python -m spacy download {SPACY_MODEL}` '
        f'ou descomente a celula de pip install no topo do notebook'
    ) from e

def _doc_to_parse(doc) -> list[list[dict]]:
    """Converte um Doc spaCy para a estrutura usada pelas regras: lista de sentencas,
    cada uma lista de dicts com id (1-indexado local), form, lemma, upos, head, deprel, start, end."""
    sentences: list[list[dict]] = []
    for sent in doc.sents:
        local_id = {tok.i: idx + 1 for idx, tok in enumerate(sent)}
        toks: list[dict] = []
        for tok in sent:
            head_id = 0 if tok.head.i == tok.i else local_id.get(tok.head.i, 0)
            toks.append({
                'id': local_id[tok.i],
                'form': tok.text,
                'lemma': (tok.lemma_ or tok.text).lower(),
                'upos': tok.pos_,
                'head': head_id,
                'deprel': tok.dep_.lower(),
                'start': tok.idx,
                'end': tok.idx + len(tok.text),
            })
        sentences.append(toks)
    return sentences

def parse_text(text: str) -> list[list[dict]]:
    """Parse de uma nota isolada (smoke test, demos, fallback no e1_extract)."""
    if not text:
        return []
    return _doc_to_parse(nlp(text))

def parse_many(records: Iterable[tuple[str, str]],
               batch_size: int = SPACY_BATCH_SIZE) -> dict[str, list[list[dict]]]:
    """Processa um lote de notas via nlp.pipe e devolve {noteId: parse}."""
    items = [(str(nid), t or '') for nid, t in records]
    if not items:
        return {}
    ids = [nid for nid, _ in items]
    texts = [t for _, t in items]
    out: dict[str, list[list[dict]]] = {}
    for note_id, doc in zip(ids, nlp.pipe(texts, batch_size=batch_size)):
        out[note_id] = _doc_to_parse(doc)
    return out

def _subtree_offsets(sent: list[dict], root_id: int) -> tuple[int, int] | None:
    """Bounding box (start, end) da subarvore enraizada em root_id."""
    members = {root_id}
    changed = True
    while changed:
        changed = False
        for tok in sent:
            if tok['head'] in members and tok['id'] not in members:
                members.add(tok['id'])
                changed = True
    offs = [(t['start'], t['end']) for t in sent if t['id'] in members and t['start'] is not None]
    if not offs:
        return None
    return min(o[0] for o in offs), max(o[1] for o in offs)

def _sentence_offsets(sent: list[dict]) -> tuple[int, int] | None:
    offs = [(t['start'], t['end']) for t in sent if t['start'] is not None]
    if not offs:
        return None
    return min(o[0] for o in offs), max(o[1] for o in offs)

In [ ]:
def lemmatize_text(text: str) -> list[str]:
    """Lematização sem parse sintático — usada para vetorizar tweets e CLAIMs no §11."""
    if not text:
        return []
    doc = nlp(text)
    return [tok.lemma_.lower() for tok in doc if tok.is_alpha and not tok.is_stop]

def lemmatize_many(texts: Iterable[str], batch_size: int = SPACY_BATCH_SIZE) -> list[list[str]]:
    items = list(texts)
    if not items:
        return []
    out = []
    for doc in nlp.pipe(items, batch_size=batch_size):
        out.append([tok.lemma_.lower() for tok in doc if tok.is_alpha and not tok.is_stop])
    return out

## 7. Estratégia E1 — regras léxico-sintáticas

In [ ]:
import re

SOURCE_DEPS = {'obl', 'nmod', 'appos', 'conj'}
SOURCE_HEAD_LEMMAS = {'segundo', 'conforme', 'acordo'}
EVIDENCE_LEMMAS = {'mostrar', 'indicar', 'comprovar', 'provar', 'evidenciar', 'confirmar', 'revelar', 'dizer'}
QUALIFIER_TERMS = {
    'provavelmente', 'possivelmente', 'supostamente', 'alegadamente', 'talvez',
    'pode', 'podem', 'poderia', 'poderiam', 'aparentemente'
}

STRONG_CONNECTOR_RE = re.compile(r'(?:\.\s+|;\s+|\n\n)')
LEADING_MARKER_RE = re.compile(r'^(?:que|de\s+que|a\s+alega[cç][aã]o\s+de\s+que)\s+', re.I)
TOKEN_RE = re.compile(r'\w+', re.UNICODE)

CLAIM_TRIGGER_PATTERNS = [
    re.compile(r'n[ãa]o\s+é\s+verdade\s+que\s+(?P<claim>.+)', re.I | re.S),
    re.compile(r'é\s+falso\s+(?:que|afirmar\s+que|dizer\s+que)\s+(?P<claim>.+)', re.I | re.S),
    re.compile(r'é\s+enganoso\s+(?:que|afirmar\s+que|dizer\s+que)\s+(?P<claim>.+)', re.I | re.S),
    re.compile(r'(?:a\s+)?(?:foto|imagem|v[íi]deo|grava[çc][ãa]o)\s+n[ãa]o\s+(?:mostra|é|foi)\s+(?P<claim>.+)', re.I | re.S),
    re.compile(r'n[ãa]o\s+h[áa]\s+(?:evid[êe]ncia|prova|registro)s?\s+de\s+que\s+(?P<claim>.+)', re.I | re.S),
    re.compile(r'n[ãa]o\s+existe\s+(?P<claim>.+)', re.I | re.S),
    re.compile(r'(?P<claim>.+?)\s+n[ãa]o\s+aconteceu\b', re.I | re.S),
    re.compile(r'(?P<claim>.+?)\s+é\s+(?:antig[oa]|falsa|de\s+\d{4})\b', re.I | re.S),
    re.compile(r'trata-se\s+de\s+(?P<claim>.+?)[,.]', re.I | re.S),
]

NEGATION_FORMS = {'não', 'nao', 'nunca'}
COMPLEMENT_DEPS = {'obj', 'ccomp', 'xcomp', 'acl', 'advcl', 'obl'}


def _sentence_like_spans(text: str):
    if not text:
        return
    start = 0
    for m in re.finditer(r'(?<=[.!?])\s+|\n{2,}', text):
        end = m.start()
        if end > start:
            yield start, end, text[start:end]
        start = m.end()
    if start < len(text):
        yield start, len(text), text[start:]


def _trim_bounds(text: str, start: int, end: int) -> tuple[int, int]:
    while start < end and text[start].isspace():
        start += 1
    while end > start and text[end - 1].isspace():
        end -= 1
    while end > start and text[end - 1] in '.,;:)]}':
        end -= 1
    return start, end


def _truncate_claim_bounds(text: str, start: int, end: int) -> tuple[int, int]:
    snippet = text[start:end]
    m = STRONG_CONNECTOR_RE.search(snippet)
    if m:
        end = start + m.start()
    return _trim_bounds(text, start, end)


def _remove_leading_markers(text: str, start: int, end: int) -> tuple[int, int]:
    local = text[start:end]
    m = LEADING_MARKER_RE.match(local)
    if m:
        start += m.end()
    return _trim_bounds(text, start, end)


def _is_valid_claim(text: str, start: int, end: int) -> bool:
    if end <= start:
        return False
    return len(TOKEN_RE.findall(text[start:end])) >= 3


def _dedupe_spans(spans):
    seen = set()
    out = []
    for span in sorted(spans, key=lambda s: (s[0], s[1], s[2])):
        if span in seen:
            continue
        seen.add(span)
        out.append(span)
    return out


def _claim_spans_from_patterns(text: str):
    spans = []
    for seg_start, _, segment in _sentence_like_spans(text):
        for pat in CLAIM_TRIGGER_PATTERNS:
            for m in pat.finditer(segment):
                start = seg_start + m.start('claim')
                end = seg_start + m.end('claim')
                start, end = _truncate_claim_bounds(text, start, end)
                start, end = _remove_leading_markers(text, start, end)
                if _is_valid_claim(text, start, end):
                    spans.append((start, end, 'CLAIM'))
    return spans


def _children_by_head(sent: list[dict]) -> dict[int, list[dict]]:
    children: dict[int, list[dict]] = {}
    for tok in sent:
        children.setdefault(int(tok.get('head', 0)), []).append(tok)
    return children


def _tok_form(tok: dict) -> str:
    return str(tok.get('form') or '').lower()


def _tok_lemma(tok: dict) -> str:
    return str(tok.get('lemma') or '').lower()


def _claim_spans_from_negated_parse(parse, text: str):
    spans = []
    if not parse:
        return spans
    for sent in parse:
        children = _children_by_head(sent)
        for tok in sent:
            if str(tok.get('upos', '')).upper() not in {'VERB', 'AUX'}:
                continue
            tok_children = children.get(int(tok.get('id', 0)), [])
            has_negation = any(_tok_form(ch) in NEGATION_FORMS or _tok_lemma(ch) in NEGATION_FORMS for ch in tok_children)
            if not has_negation:
                continue
            complements = [
                ch for ch in tok_children
                if str(ch.get('deprel', '')).lower().split(':')[0] in COMPLEMENT_DEPS
            ]
            if not complements:
                continue
            complements.sort(key=lambda ch: int(ch.get('start') or 10**9))
            off = _subtree_offsets(sent, int(complements[0]['id']))
            if not off:
                continue
            start, end = _truncate_claim_bounds(text, off[0], off[1])
            start, end = _remove_leading_markers(text, start, end)
            if _is_valid_claim(text, start, end):
                spans.append((start, end, 'CLAIM'))
    return spans


def rules_evidencia_from_parse(parse, fontes=None):
    """EVIDENCIA fundida: verbo factivo (v2) + sinal numérico não dominado por FONTE (v1)."""
    fontes = fontes or []
    spans = []

    # Sinal 1 (v2): sentenças com verbo factivo / de evidência
    for sent in parse:
        for tok in sent:
            if str(tok.get('upos', '')).upper() == 'VERB' and _tok_lemma(tok) in EVIDENCE_LEMMAS:
                off = _subtree_offsets(sent, int(tok['id']))
                if off:
                    spans.append(Span(off[0], off[1], 'EVIDENCIA'))

    # Sinal 2 (v1): sentenças com NUM/% não dominadas por FONTE
    for sent in parse:
        has_num = any(
            str(t.get('upos', '')).upper() == 'NUM' or '%' in (t.get('form') or '')
            for t in sent
        )
        if not has_num:
            continue
        soff = _sentence_offsets(sent)
        if not soff:
            continue
        slen = max(1, soff[1] - soff[0])
        dominated = any(
            max(0, min(soff[1], f.end) - max(soff[0], f.start)) / slen > 0.5
            for f in fontes
        )
        if not dominated:
            spans.append(Span(soff[0], soff[1], 'EVIDENCIA'))

    return spans


def rules_claim_from_parse(parse, text: str | None = None):
    """CLAIM v2: padroes de negacao/contraste + complemento sintatico barato."""
    if text is None:
        text = ''
    spans = []
    spans.extend(_claim_spans_from_patterns(text))
    spans.extend(_claim_spans_from_negated_parse(parse, text))
    return [Span(start, end, typ) for start, end, typ in _dedupe_spans(spans)]


In [ ]:
LETTER = r'[^\W\d_]'
RE_FONTE_PREP = re.compile(
    r'\b(?:segundo|de\s+acordo\s+com|conforme)\s+(?:o|a|os|as|um|uma)?\s*'
    rf'({LETTER}[\w\s\-\']{{2,50}})',
    re.IGNORECASE,
)
RE_FONTE_VERBO = re.compile(
    rf'({LETTER}[\w\s\-\']{{2,50}})\s+'
    r'(?:afirma|afirmou|disse|diz|declarou|declara|publicou|publica|noticiou|noticia|informou|informa|apurou|apura|relatou|relata)\b',
    re.IGNORECASE,
)
RE_FONTE_URL = re.compile(r'https?://\S+')

RE_QUAL = re.compile(
    r'\b(parcialmente|fora\s+de\s+contexto|sem\s+evid[êe]ncia(?:s)?|supostamente|'
    r'ao\s+que\s+tudo\s+indica|n[ãa]o\s+h[áa]\s+prova(?:s)?|aparentemente|alegadamente)\b',
    re.IGNORECASE,
)

def e1_extract(text: str, note_id: str | None = None,
               parse: list[list[dict]] | None = None) -> list[Span]:
    spans: list[Span] = []
    # FONTE via entidades pre-extraidas do HF: preserva URLs removidas de summary_clean.
    if note_id is not None:
        spans.extend(ENTITY_SOURCE_SPANS.get(str(note_id), []))
    # FONTE via regex complementar.
    for rx in (RE_FONTE_PREP, RE_FONTE_VERBO, RE_FONTE_URL):
        for m in rx.finditer(text):
            spans.append(Span(m.start(), m.end(), 'FONTE'))
    # QUALIFICADOR via regex
    for m in RE_QUAL.finditer(text):
        spans.append(Span(m.start(), m.end(), 'QUALIFICADOR'))
    # CLAIM e EVIDENCIA via dependency parsing (spaCy)
    if text:
        if parse is None:
            parse = parse_text(text)
        if parse:
            spans.extend(rules_claim_from_parse(parse, text=text))
            fonte_so_far = [s for s in spans if s.type == 'FONTE']
            spans.extend(rules_evidencia_from_parse(parse, fonte_so_far))
    return normalize_spans(spans, text)

# sanity check
demo = 'Segundo a Agência Brasil, o reajuste foi de 4,9% e não 500%. Veja https://agenciabrasil.ebc.com.br/exemplo. O número está parcialmente correto.'
for s in e1_extract(demo, note_id='__demo__'):
    print(s, '->', repr(s.text(demo)))

### 7.1 Cobertura E1 (sanity check)

In [ ]:
# Sanity check isolado do E1: cobertura por tipo no subset final.
e1_records = list(subset[['noteId', '_text']].itertuples(index=False, name=None))
parses_for_e1 = parse_many(e1_records, batch_size=64)
e1_cov_payloads = [
    e1_extract(text, note_id=note_id, parse=parses_for_e1.get(str(note_id), []))
    for note_id, text in e1_records
]

e1_coverage = {}
for typ in LABELS:
    e1_coverage[typ] = sum(any(span.type == typ for span in spans) for spans in e1_cov_payloads) / max(len(e1_cov_payloads), 1)

e1_coverage_df = (
    pd.DataFrame({'type': list(e1_coverage), 'coverage': list(e1_coverage.values())})
      .sort_values('type')
      .reset_index(drop=True)
)
print('Cobertura E1 snippet_v2:')
display(e1_coverage_df)
print(f"Cobertura CLAIM E1: {e1_coverage.get('CLAIM', 0):.1%} (criterio interno: >= 30%)")


## 8. Estratégia E2 — LLM (Qwen via DashScope)

### 8.1 Cliente

In [ ]:
import os
from getpass import getpass
from openai import OpenAI

QWEN_MODEL = 'qwen3.6-max-preview'
QWEN_BASE_URL = os.getenv('QWEN_BASE_URL', 'https://dashscope-intl.aliyuncs.com/compatible-mode/v1')

QWEN_API_KEY = os.getenv('QWEN_API_KEY') or os.getenv('DASHSCOPE_API_KEY') or os.getenv('ALIBABA_API_KEY')

if not QWEN_API_KEY:
    try:
        from google.colab import userdata
        QWEN_API_KEY = userdata.get('QWEN_API')
    except Exception:
        pass

QWEN_TIMEOUT = 120.0
QWEN_MAX_RETRIES = 3
ENABLE_THINKING = False

if not QWEN_API_KEY:
    QWEN_API_KEY = getpass('Cole sua QWEN_API_KEY/DASHSCOPE_API_KEY: ')

qwen_client = OpenAI(
    api_key=QWEN_API_KEY,
    base_url=QWEN_BASE_URL,
    max_retries=QWEN_MAX_RETRIES,
    timeout=QWEN_TIMEOUT,
)

print('Cliente Qwen configurado:', QWEN_MODEL, '| base_url:', QWEN_BASE_URL)
print('enable_thinking:', ENABLE_THINKING, '| timeout:', QWEN_TIMEOUT, '| max_retries:', QWEN_MAX_RETRIES)

### 8.2 Prompt, few-shot, alinhamento snippet→offset, `e2_extract`

Quatro níveis de alinhamento em cascata: `exact` → `normalized` (espaços) → `unicode_normalized` (NFKC + aspas/travessões) → `regex` (palavras com separadores `\W+`).

In [ ]:
import json
import time
import unicodedata
from collections import Counter

VALID_TYPES = {'CLAIM', 'EVIDENCIA', 'FONTE', 'QUALIFICADOR'}

PROMPT_SYSTEM = '''Você é um anotador de mineração de argumentos em notas de contexto brasileiras.
Extraia spans argumentativos copiando trechos LITERALMENTE do texto original.
NUNCA invente texto, NUNCA parafraseie, NUNCA devolva offsets.
Responda apenas JSON válido no formato {"spans": [{"type": "CLAIM|EVIDENCIA|FONTE|QUALIFICADOR", "text": "trecho exato"}]}.
Tipos:
- CLAIM: alegacao que a nota corrige, nega, relativiza ou contradiz. Em notas de checagem, frequentemente aparece depois de "não é verdade que", "é falso que", "não há evidência de que", "imagem não mostra".
- EVIDENCIA: dado, verificacao ou justificativa usada para sustentar a correcao.
- FONTE: fonte citada, veiculo, orgao, URL, documento ou base consultada.
- QUALIFICADOR: marcador de incerteza, ressalva, escopo ou modalidade.
Regras criticas de copia:
- O campo text deve ser uma substring literal do texto original.
- Preserve acentos, aspas curvas, travessoes, hifens e capitalizacao exatamente como aparecem.
- Se o texto tem quebras de linha (\n) dentro de um trecho, NAO inclua as quebras no snippet: copie apenas o conteudo de uma frase continua.
- Se uma URL aparece no texto, ela sera extraida automaticamente como FONTE pelo pipeline; ainda assim, marque outras fontes textuais relevantes.
- Se nao houver span argumentativo claro, retorne {"spans": []}.
'''

FEW_SHOT = [
    {'role': 'user', 'content': 'TEXTO: Segundo a Agência Lupa, é falso que vacinas alterem o DNA. Estudos clínicos mostram que elas não entram no núcleo das células. Fonte: https://lupa.uol.com.br/'},
    {'role': 'assistant', 'content': json.dumps({'spans': [
        {'type': 'FONTE', 'text': 'Segundo a Agência Lupa'},
        {'type': 'CLAIM', 'text': 'vacinas alterem o DNA'},
        {'type': 'EVIDENCIA', 'text': 'Estudos clínicos mostram que elas não entram no núcleo das células'},
        {'type': 'FONTE', 'text': 'https://lupa.uol.com.br/'}
    ]}, ensure_ascii=False)},
    {'role': 'user', 'content': 'TEXTO: O vídeo provavelmente foi gravado em 2019. De acordo com a Reuters, as imagens são de uma manifestação no Chile, não de um ato recente no Brasil.'},
    {'role': 'assistant', 'content': json.dumps({'spans': [
        {'type': 'QUALIFICADOR', 'text': 'provavelmente'},
        {'type': 'FONTE', 'text': 'De acordo com a Reuters'},
        {'type': 'EVIDENCIA', 'text': 'as imagens são de uma manifestação no Chile'},
        {'type': 'CLAIM', 'text': 'um ato recente no Brasil'}
    ]}, ensure_ascii=False)},
    {'role': 'user', 'content': 'TEXTO: Obrigado pela sugestão. A comunidade deve manter o debate respeitoso.'},
    {'role': 'assistant', 'content': json.dumps({'spans': []}, ensure_ascii=False)},
    {'role': 'user', 'content': 'TEXTO: 1: O vídeo não mostra fraude em urnas no Brasil.\n\n2: A gravação foi publicada em 2018 e mostra uma simulação escolar, segundo o TSE.'},
    {'role': 'assistant', 'content': json.dumps({'spans': [
        {'type': 'CLAIM', 'text': 'fraude em urnas no Brasil'},
        {'type': 'EVIDENCIA', 'text': 'A gravação foi publicada em 2018 e mostra uma simulação escolar'},
        {'type': 'FONTE', 'text': 'segundo o TSE'}
    ]}, ensure_ascii=False)},
    {'role': 'user', 'content': 'TEXTO: O post atribui a fala “vamos fechar o Congresso” ao ministro. Segundo o STF, a citação é falsa.'},
    {'role': 'assistant', 'content': json.dumps({'spans': [
        {'type': 'CLAIM', 'text': '“vamos fechar o Congresso”'},
        {'type': 'FONTE', 'text': 'Segundo o STF'},
        {'type': 'EVIDENCIA', 'text': 'a citação é falsa'}
    ]}, ensure_ascii=False)},
    {'role': 'user', 'content': 'TEXTO: A imagem não mostra enchente no RS em 2024. Ela é de 2011, no Japão, conforme checagem da AFP.'},
    {'role': 'assistant', 'content': json.dumps({'spans': [
        {'type': 'CLAIM', 'text': 'enchente no RS em 2024'},
        {'type': 'EVIDENCIA', 'text': 'Ela é de 2011, no Japão'},
        {'type': 'FONTE', 'text': 'conforme checagem da AFP'}
    ]}, ensure_ascii=False)},
]


def _safe_json_loads(content: str):
    try:
        return json.loads(content)
    except json.JSONDecodeError:
        m = re.search(r'\{.*\}', content, re.S)
        if not m:
            raise
        return json.loads(m.group(0))


def _normalize_spaces(s: str) -> str:
    return re.sub(r'\s+', ' ', s).strip()


def _overlaps_used(start: int, end: int, used: list[tuple[int, int]]) -> bool:
    return any(not (end <= a or start >= b) for a, b in used)


def _find_literal(text: str, snippet: str, used: list[tuple[int, int]]):
    if not snippet:
        return None
    pos = text.find(snippet)
    while pos >= 0:
        end = pos + len(snippet)
        if not _overlaps_used(pos, end, used):
            return pos, end
        pos = text.find(snippet, pos + 1)
    return None


def _find_space_normalized(text: str, snippet: str, used: list[tuple[int, int]]):
    norm_snippet = _normalize_spaces(snippet)
    if not norm_snippet:
        return None
    chunks = []
    mapping = []
    in_space = False
    for i, ch in enumerate(text):
        if ch.isspace():
            if not in_space:
                chunks.append(' ')
                mapping.append(i)
                in_space = True
        else:
            chunks.append(ch)
            mapping.append(i)
            in_space = False
    full_norm = ''.join(chunks)
    left_trim = len(full_norm) - len(full_norm.lstrip())
    norm_text = full_norm.strip()
    if left_trim:
        mapping = mapping[left_trim:]
    pos = norm_text.find(norm_snippet)
    while pos >= 0:
        start = mapping[pos]
        end = mapping[pos + len(norm_snippet) - 1] + 1
        if not _overlaps_used(start, end, used):
            return start, end
        pos = norm_text.find(norm_snippet, pos + 1)
    return None


UNICODE_TRANSLATION = str.maketrans({
    '“': '"', '”': '"', '„': '"', '«': '"', '»': '"',
    '‘': "'", '’': "'", '‚': "'",
    '–': '-', '—': '-', '−': '-',
    '\u00a0': ' ',
})


def _normalize_unicode_with_map(s: str):
    out = []
    mapping = []
    for i, ch in enumerate(s):
        translated = ch.translate(UNICODE_TRANSLATION)
        normalized = unicodedata.normalize('NFKC', translated)
        for out_ch in normalized:
            out.append(out_ch)
            mapping.append(i)
    return ''.join(out), mapping


def _find_unicode_normalized(text: str, snippet: str, used: list[tuple[int, int]]):
    norm_text, mapping = _normalize_unicode_with_map(text)
    norm_snippet, _ = _normalize_unicode_with_map(snippet)
    norm_snippet_spaces = _normalize_spaces(norm_snippet)

    for needle in [norm_snippet, norm_snippet_spaces]:
        if not needle:
            continue
        pos = norm_text.find(needle)
        while pos >= 0:
            start = mapping[pos]
            end = mapping[pos + len(needle) - 1] + 1
            if not _overlaps_used(start, end, used):
                return start, end
            pos = norm_text.find(needle, pos + 1)

    compact = []
    compact_map = []
    in_space = False
    for j, ch in enumerate(norm_text):
        if ch.isspace():
            if not in_space:
                compact.append(' ')
                compact_map.append(mapping[j])
                in_space = True
        else:
            compact.append(ch)
            compact_map.append(mapping[j])
            in_space = False
    full_compact = ''.join(compact)
    left_trim = len(full_compact) - len(full_compact.lstrip())
    compact_text = full_compact.strip()
    if left_trim:
        compact_map = compact_map[left_trim:]
    pos = compact_text.find(norm_snippet_spaces)
    while pos >= 0:
        start = compact_map[pos]
        end = compact_map[pos + len(norm_snippet_spaces) - 1] + 1
        if not _overlaps_used(start, end, used):
            return start, end
        pos = compact_text.find(norm_snippet_spaces, pos + 1)
    return None


def _find_regex_relaxed(text: str, snippet: str, used: list[tuple[int, int]]):
    words = re.findall(r'\w+', snippet, flags=re.UNICODE)
    if len(words) < 3:
        return None
    pattern = r'\W+'.join(re.escape(w) for w in words[:12])
    try:
        regex = re.compile(pattern, re.I | re.S)
    except re.error:
        return None
    for m in regex.finditer(text):
        if not _overlaps_used(m.start(), m.end(), used):
            return m.start(), m.end()
    return None


def align_span_text(text: str, snippet: str, used: list[tuple[int, int]] | None = None):
    used = used or []
    for level, finder in [
        ('exact', _find_literal),
        ('normalized', _find_space_normalized),
        ('unicode_normalized', _find_unicode_normalized),
        ('regex', _find_regex_relaxed),
    ]:
        found = finder(text, snippet, used)
        if found:
            return {'start': found[0], 'end': found[1], 'level': level}
    return {'start': None, 'end': None, 'level': 'failed'}


def _qwen_call(text: str):
    messages = [{'role': 'system', 'content': PROMPT_SYSTEM}] + FEW_SHOT + [{'role': 'user', 'content': f'TEXTO: {text}'}]
    return qwen_client.chat.completions.create(
        model=QWEN_MODEL,
        messages=messages,
        temperature=0,
        seed=RANDOM_SEED,
        response_format={'type': 'json_object'},
        timeout=QWEN_TIMEOUT,
        extra_body={'enable_thinking': ENABLE_THINKING},
    )


def e2_extract(text: str, capture_reasoning: bool = False):
    started = time.perf_counter()
    result = {
        'spans': [],
        'error': None,
        'attempts': 1,
        'latency_s': None,
        'raw': None,
        'align_level_counts': {'exact': 0, 'normalized': 0, 'unicode_normalized': 0, 'regex': 0, 'failed': 0},
        'reasoning': None,
    }
    try:
        response = _qwen_call(text)
        message = response.choices[0].message
        raw = message.content or '{}'
        result['raw'] = raw
        if capture_reasoning:
            result['reasoning'] = getattr(message, 'reasoning_content', None) or getattr(message, 'reasoning', None)
        payload = _safe_json_loads(raw)
        spans = payload.get('spans', []) if isinstance(payload, dict) else []
        used = []
        aligned = []
        for item in spans:
            typ = str(item.get('type', '')).upper().strip()
            snippet = str(item.get('text', ''))
            if typ not in VALID_TYPES or not snippet:
                continue
            match = align_span_text(text, snippet, used=used)
            level = match['level']
            result['align_level_counts'][level] = result['align_level_counts'].get(level, 0) + 1
            if level == 'failed':
                aligned.append({'start': None, 'end': None, 'type': typ, 'text': snippet, 'align_level': level})
                continue
            used.append((match['start'], match['end']))
            aligned.append({
                'start': match['start'],
                'end': match['end'],
                'type': typ,
                'text': text[match['start']:match['end']],
                'snippet_requested': snippet,
                'align_level': level,
            })
        result['spans'] = aligned
    except Exception as exc:
        result['error'] = f'{type(exc).__name__}: {exc}'
    finally:
        result['latency_s'] = time.perf_counter() - started
    return result


print('E2 configurado para', QWEN_MODEL)

## 9. Métricas span-level (F1 strict, F1 relaxed, κ char-level)

In [ ]:
def _overlap(a: Span, b: Span) -> int:
    return max(0, min(a.end, b.end) - max(a.start, b.start))

def f1_strict(gold: list[Span], pred: list[Span]) -> float:
    gset = {(s.start, s.end, s.type) for s in gold}
    pset = {(s.start, s.end, s.type) for s in pred}
    tp = len(gset & pset)
    if not tp:
        return 0.0 if (gset or pset) else 1.0
    p = tp / len(pset)
    r = tp / len(gset)
    return 2 * p * r / (p + r)

def f1_relaxed(gold: list[Span], pred: list[Span], min_overlap_ratio: float = 0.5) -> float:
    matched_g, matched_p = set(), set()
    for i, g in enumerate(gold):
        for j, p in enumerate(pred):
            if g.type != p.type or j in matched_p:
                continue
            ov = _overlap(g, p)
            denom = max(g.end - g.start, p.end - p.start, 1)
            if ov / denom >= min_overlap_ratio:
                matched_g.add(i)
                matched_p.add(j)
                break
    tp = len(matched_g)
    if not tp:
        return 0.0 if (gold or pred) else 1.0
    p = tp / len(pred) if pred else 0.0
    r = tp / len(gold) if gold else 0.0
    return 2 * p * r / (p + r) if (p + r) else 0.0

def _char_labels(text: str, spans: list[Span]) -> list[str]:
    labels = ['O'] * len(text)
    for s in spans:
        for i in range(s.start, min(s.end, len(text))):
            labels[i] = s.type
    return labels

def cohen_kappa_chars(text: str, a: list[Span], b: list[Span]) -> float:
    if not text:
        return 0.0
    la = _char_labels(text, a)
    lb = _char_labels(text, b)
    n = len(text)
    po = sum(1 for x, y in zip(la, lb) if x == y) / n
    cats = set(la) | set(lb)
    pe = sum((la.count(c) / n) * (lb.count(c) / n) for c in cats)
    return (po - pe) / (1 - pe) if pe < 1 else 1.0

## 10. Classificador de meta-nota

Move o `is_meta` para **antes** do sweep/seleção. Usa regex sobre o texto da nota: prefixo `NNN`, padrões fixos ("não necessita nota", "claramente piada", "postagem de humor", "uso indevido") e fallback de comprimento mínimo. A flag é aplicada ao `sweep` no §12 e usada como filtro em §14 (seleção qualitativa).

In [ ]:
import re

META_PREFIX_RE = re.compile(r'^\s*NNN\b', re.I)
META_PATTERNS = [
    (re.compile(r'\bn[ãa]o\s+(necessita|precisa|[ée]\s+necess[áa]ria)\s+(de\s+)?notas?\b', re.I),
     'nao_necessita_nota'),
    (re.compile(r'\b(uso\s+(indevido|inadequado)|utiliz(em|ar)\s+as?\s+notas?\s+da\s+comunidade)\b', re.I),
     'uso_indevido_nota'),
    (re.compile(r'\b(claramente\s+(uma?\s+)?)?(piada|s[áa]tira|opini[ãa]o\s+pessoal)\b\.?\s*$', re.I),
     'piada_satira_opiniao'),
    (re.compile(r'\bpost(agem)?\s+(de\s+)?humor\b', re.I), 'humor'),
]
META_SHORT_CHARS = 50

def classify_meta(text: str) -> tuple[bool, str]:
    """Retorna (is_meta, reason). Aplicar sobre o texto da nota."""
    if not isinstance(text, str) or not text.strip():
        return True, 'vazio'
    if META_PREFIX_RE.search(text):
        return True, 'prefixo_NNN'
    for rx, lab in META_PATTERNS:
        if rx.search(text):
            return True, lab
    if len(re.sub(r'\s+', '', text)) < META_SHORT_CHARS:
        return True, 'muito_curta'
    return False, ''

# sanity
for demo in [
    'NNN- post de humor, sem pretensão informativa',
    'A foto não mostra protesto em 2024, é de 2013 segundo a Lupa',
    'piada',
]:
    print(repr(demo[:60]), '->', classify_meta(demo))

## 11. Smoke test (10 notas)

In [ ]:
smoke = subset.head(10).copy()
rows = []
for _, row in smoke.iterrows():
    note_id = row['noteId']
    text = row['_text']
    e1 = e1_extract(text, note_id=note_id)
    e2_res = e2_extract(text)

    # Converte dicionários do E2 para objetos Span, ignorando falhas de alinhamento
    e2 = [Span(s['start'], s['end'], s['type']) for s in e2_res['spans'] if s.get('start') is not None]

    rows.append({
        'noteId': note_id,
        'len_text': len(text),
        'n_e1': len(e1),
        'n_e2': len(e2),
        'f1_strict': round(f1_strict(e1, e2), 3),
        'f1_relaxed': round(f1_relaxed(e1, e2), 3),
        'kappa': round(cohen_kappa_chars(text, e1, e2), 3),
        'e2_err': e2_res['error'],
        'e2_align': e2_res['align_level_counts'],
        'e2_attempts': e2_res['attempts'],
    })

pd.DataFrame(rows)

## 12. Sweep completo (E1 + E2)

Grava `sweep_final.jsonl` em modo append, com resume por noteId. Cada linha contém ambos os spans, latências, contagem de níveis de alinhamento do E2 e mensagem de erro se houve.

In [ ]:
def _read_processed_ids(path: Path, use_e2: bool = True,
                        retry_e2_errors: set[str] | None = None) -> set[str]:
    if not path.exists():
        return set()
    retry_e2_errors = retry_e2_errors or {'no_api_key', 'disabled'}
    latest: dict[str, dict] = {}
    with path.open('r', encoding='utf-8') as f:
        for line in f:
            try:
                rec = json.loads(line)
                latest[str(rec['noteId'])] = rec
            except (json.JSONDecodeError, KeyError):
                continue
    done = set()
    for note_id, rec in latest.items():
        if use_e2 and rec.get('e2_err') in retry_e2_errors:
            continue
        done.add(note_id)
    return done

def _spans_to_payload(spans) -> list[dict]:
    out = []
    for s in spans or []:
        if isinstance(s, Span):
            out.append(s.to_dict())  # type: ignore
        elif isinstance(s, dict) and {'start', 'end', 'type'} <= s.keys():
            if s['start'] is not None and s['end'] is not None:
                out.append({'start': int(s['start']), 'end': int(s['end']), 'type': str(s['type']).upper()})
    return out

def run_sweep(df: pd.DataFrame, out_path: Path = SWEEP_JSONL, use_e2: bool = True,
              flush_every: int = 25, spacy_batch_size: int = SPACY_BATCH_SIZE) -> None:
    done = _read_processed_ids(out_path, use_e2=use_e2)
    pending = df[~df['noteId'].astype(str).isin(done)].reset_index(drop=True)
    print(f'já processadas: {len(done):,}  | pendentes: {len(pending):,}  | total: {len(df):,}')
    if not len(pending):
        return
    buffer: list[str] = []
    chunk = max(1, spacy_batch_size)
    with out_path.open('a', encoding='utf-8') as f:
        try:
            pbar = tqdm(total=len(pending), desc='sweep E1+E2')
            for start in range(0, len(pending), chunk):
                batch_df = pending.iloc[start:start + chunk]
                tp0 = time.perf_counter()
                parses = parse_many(
                    ((str(row['noteId']), row['_text']) for _, row in batch_df.iterrows()),
                    batch_size=chunk,
                )
                # Custo do parse amortizado no lote (nlp.pipe): a latencia fim-a-fim
                # do E1 por nota e' e1_parse_ms + e1_ms.
                parse_ms_avg = (time.perf_counter() - tp0) * 1000 / max(len(batch_df), 1)
                for _, row in batch_df.iterrows():
                    note_id = str(row['noteId'])
                    text = row['_text']
                    t0 = time.perf_counter()
                    e1 = e1_extract(text, note_id=note_id, parse=parses.get(note_id))
                    t1 = time.perf_counter()
                    if use_e2:
                        e2_res = e2_extract(text)
                        e2 = e2_res['spans']
                        e2_err = e2_res['error']
                        e2_attempts = e2_res['attempts']
                        e2_align = e2_res['align_level_counts']
                    else:
                        e2, e2_err, e2_attempts, e2_align = [], 'disabled', 0, {}
                    t2 = time.perf_counter()
                    rec = {
                        'noteId': note_id,
                        'tweetId': str(row.get('tweetId') or ''),
                        'text': text,
                        'tweet_text': row.get('tweet_text') or '',
                        'consenso': row.get('consenso'),
                        'macrotheme_label': row.get('macrotheme_label'),
                        'e1_spans': _spans_to_payload(e1),
                        'e2_spans': _spans_to_payload(e2),
                        'e1_ms': round((t1 - t0) * 1000, 1),
                        'e1_parse_ms': round(parse_ms_avg, 2),
                        'e2_ms': round((t2 - t1) * 1000, 1),
                        'e2_err': e2_err,
                        'e2_attempts': e2_attempts,
                        'e2_align_levels': e2_align,
                        'model': QWEN_MODEL if use_e2 else None,
                    }
                    buffer.append(json.dumps(rec, ensure_ascii=False))
                    if len(buffer) >= flush_every:
                        f.write('\n'.join(buffer) + '\n'); f.flush(); buffer.clear()
                    pbar.update(1)
        finally:
            if buffer:
                f.write('\n'.join(buffer) + '\n'); f.flush()
            try:
                pbar.close()
            except NameError:
                pass

run_sweep(subset, use_e2=True)


## 13. Consolidação do sweep — loader seguro + meta-flag + agreement

Substitui `pd.read_json(..., lines=True)` por um loader manual com `json.loads` linha-a-linha, forçando `noteId` e `tweetId` como string. Isso evita o bug de precisão para snowflake IDs > 2⁵³.

Aplica `classify_meta` aqui (§10), de modo que a flag fica disponível em todos os artefatos downstream. Recalcula `agreement` no mesmo passo para garantir que o `noteId` íntegro propague.

In [ ]:
def load_sweep_safe(path: Path) -> pd.DataFrame:
    """Lê o JSONL preservando noteIds grandes como string e deduplicando por id."""
    rows: dict[str, dict] = {}
    with path.open('r', encoding='utf-8') as f:
        for ln in f:
            ln = ln.strip()
            if not ln:
                continue
            try:
                r = json.loads(ln)
            except json.JSONDecodeError:
                continue
            r['noteId']  = str(r['noteId'])
            r['tweetId'] = str(r.get('tweetId') or '')
            rows[r['noteId']] = r
    return pd.DataFrame(rows.values())

sweep = load_sweep_safe(SWEEP_JSONL)
assert sweep['noteId'].dtype == object, 'noteId precisa ser string'
print(f'sweep carregado: {len(sweep):,} notas únicas')
errs = sweep['e2_err'].dropna().value_counts() if 'e2_err' in sweep else pd.Series(dtype=int)
if len(errs):
    print('erros em E2:', dict(errs))

# aplica is_meta
flagged = sweep['text'].apply(classify_meta)
sweep['is_meta']     = flagged.str[0]
sweep['meta_reason'] = flagged.str[1]
print(f'\nmeta-notas detectadas: {int(sweep["is_meta"].sum()):,} '
      f'({100*sweep["is_meta"].mean():.1f}%)')
print(sweep[sweep['is_meta']]['meta_reason'].value_counts().to_string())

### 13.1 Agreement E1 × E2 (regravado a partir do sweep íntegro)

In [ ]:
def _from_payload(payload) -> list[Span]:
    return [Span(int(s['start']), int(s['end']), str(s['type']))
            for s in (payload or []) if s.get('start') is not None]

agree_rows = []
for r in sweep.itertuples(index=False):
    e1 = _from_payload(r.e1_spans)
    e2 = _from_payload(r.e2_spans)
    agree_rows.append({
        'noteId': r.noteId, 'tweetId': r.tweetId,
        'consenso': r.consenso, 'macrotheme_label': r.macrotheme_label,
        'is_meta': r.is_meta, 'meta_reason': r.meta_reason,
        'len_text': len(r.text or ''),
        'n_e1': len(e1), 'n_e2': len(e2),
        'f1_strict':  f1_strict(e1, e2),
        'f1_relaxed': f1_relaxed(e1, e2),
        'kappa':      cohen_kappa_chars(r.text or '', e1, e2),
    })
agreement = pd.DataFrame(agree_rows)
agreement.to_parquet(AGREEMENT_PARQUET, index=False)
print(f'agreement_final.parquet: {agreement.shape}')
print('\n--- estatísticas globais ---')
print(agreement[['f1_strict','f1_relaxed','kappa']].describe(percentiles=[.5,.95]).round(3).to_string())

### 13.2 Tabela κ tripla — A_completo / B_sem_meta / C_ambos_marcaram

In [ ]:
def _pres(payload, typ):
    return any(s.get('type') == typ for s in (payload or []))

def kappa_presence(a, b):
    n = len(a)
    if n == 0: return float('nan')
    po = (a == b).mean()
    pe = (a.sum()/n)*(b.sum()/n) + ((n-a.sum())/n)*((n-b.sum())/n)
    return (po - pe) / (1 - pe) if pe < 1 else 0.0

def block(df, label):
    out = []
    for t in LABELS:
        a = df['e1_spans'].apply(lambda p: _pres(p, t))
        b = df['e2_spans'].apply(lambda p: _pres(p, t))
        out.append({'corte': label, 'n': len(df), 'tipo': t,
                    'cob_E1_%': round(100*a.mean(), 1),
                    'cob_E2_%': round(100*b.mean(), 1),
                    'kappa': round(kappa_presence(a, b), 3)})
    return out

n_e1 = sweep['e1_spans'].apply(lambda p: len(p) if isinstance(p, list) else 0)
n_e2 = sweep['e2_spans'].apply(lambda p: len(p) if isinstance(p, list) else 0)

tabela = pd.DataFrame(
    block(sweep,                          'A_completo')
  + block(sweep[~sweep['is_meta']],       'B_sem_meta')
  + block(sweep[(n_e1>0) & (n_e2>0)],     'C_ambos_marcaram')
)
print(tabela.to_string(index=False))
print('\n----- κ por tipo, 3 cortes -----')
print(tabela.pivot(index='tipo', columns='corte', values='kappa')
            .reindex(LABELS)[['A_completo','B_sem_meta','C_ambos_marcaram']]
            .to_string())

## 14. Análise tweet-aware — fidelidade do CLAIM ao tweet original

Cosseno TF-IDF entre lemas do CLAIM extraído (E1 e E2) e do tweet original. Métrica orientativa: CLAIMs com baixa similaridade são candidatos a falso positivo (a estratégia inventou alegação que não corresponde ao tweet checado).

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

def _lemma_str(lemmas: list[str]) -> str:
    return ' '.join(lemmas)

# Lematiza tweets e CLAIMs em batch
tweet_texts = sweep['tweet_text'].fillna('').tolist()
print('lematizando tweets...')
tweet_lemmas = lemmatize_many(tweet_texts)
sweep['_tweet_lem'] = [_lemma_str(l) for l in tweet_lemmas]

def _strategy_claim_lemmas(col):
    # Extrai o texto do CLAIM usando os offsets no texto original
    lists_of_claim_lists = sweep.apply(
        lambda r: [r['text'][s['start']:s['end']]
                   for s in (r[col] or [])
                   if s.get('type') == 'CLAIM'],
        axis=1
    )
    flat_claims, owner_idx = [], []
    for i, claims in enumerate(lists_of_claim_lists):
        for c in claims:
            flat_claims.append(c); owner_idx.append(i)
    print(f'  {col}: {len(flat_claims)} CLAIMs em {sum(1 for x in lists_of_claim_lists if x)} notas')
    if not flat_claims:
        return pd.DataFrame(columns=['row_idx', 'claim_text', 'claim_lem', 'tweet_lem', 'sim'])
    lemmas = lemmatize_many(flat_claims)
    return pd.DataFrame({
        'row_idx': owner_idx,
        'claim_text': flat_claims,
        'claim_lem': [_lemma_str(l) for l in lemmas],
        'tweet_lem': [sweep.iloc[i]['_tweet_lem'] for i in owner_idx],
    })

print('lematizando CLAIMs E1...')
df_e1 = _strategy_claim_lemmas('e1_spans')
print('lematizando CLAIMs E2...')
df_e2 = _strategy_claim_lemmas('e2_spans')

def _cosine_per_row(df):
    if df.empty:
        df['sim'] = pd.Series(dtype=float); return df
    sims = []
    for _, r in df.iterrows():
        if not r['claim_lem'].strip() or not r['tweet_lem'].strip():
            sims.append(0.0); continue
        v = TfidfVectorizer().fit_transform([r['claim_lem'], r['tweet_lem']])
        sims.append(float(cosine_similarity(v[0], v[1])[0, 0]))
    df = df.copy()
    df['sim'] = sims
    return df

df_e1 = _cosine_per_row(df_e1)
df_e2 = _cosine_per_row(df_e2)


In [ ]:
# Agrega: similaridade média do CLAIM ao tweet, por estratégia e por consenso
def _agg(df, strat):
    if df.empty:
        return pd.DataFrame(columns=['estrategia', 'n_claims', 'sim_mean', 'sim_p25', 'sim_p50', 'sim_p75'])
    return pd.DataFrame([{
        'estrategia': strat,
        'n_claims': len(df),
        'sim_mean': df['sim'].mean(),
        'sim_p25': df['sim'].quantile(.25),
        'sim_p50': df['sim'].quantile(.50),
        'sim_p75': df['sim'].quantile(.75),
    }])

global_table = pd.concat([_agg(df_e1, 'E1'), _agg(df_e2, 'E2')], ignore_index=True)
print('--- fidelidade média do CLAIM ao tweet original ---')
print(global_table.round(3).to_string(index=False))

# Por consenso
def _agg_consenso(df, strat):
    if df.empty:
        return pd.DataFrame()
    tmp = df.copy()
    tmp['consenso'] = tmp['row_idx'].map(lambda i: sweep.iloc[i]['consenso'])
    return tmp.groupby('consenso')['sim'].agg(['count', 'mean', 'median']).reset_index().assign(estrategia=strat)

by_cons = pd.concat([_agg_consenso(df_e1, 'E1'), _agg_consenso(df_e2, 'E2')], ignore_index=True)
print('\n--- por consenso ---')
print(by_cons.round(3).to_string(index=False))

# Salva detalhe por CLAIM para uso na seleção qualitativa / inspeção
df_e1['estrategia'] = 'E1'; df_e2['estrategia'] = 'E2'
align_long = pd.concat([df_e1, df_e2], ignore_index=True)
align_long.to_parquet(TWEET_ALIGN_PARQUET, index=False)
print(f'\nsalvo {TWEET_ALIGN_PARQUET}')

## 15. Seleção qualitativa — 60 notas (20 por consenso, sem meta)

Filtra `~is_meta` antes do `_select_within` para garantir que nenhuma meta-nota entre na amostra. O pool por consenso fica menor — se ainda não bater 60, o filler interno completa do mesmo pool.

In [ ]:
# === Amostra qualitativa CONGELADA (60 notas ja anotadas por humanos) ===
# A selecao original (celula seguinte) usa metricas E1xE2 como criterio. Como essas
# metricas mudam quando o E1 e' corrigido (ver §19), re-executar a selecao escolheria
# OUTRAS notas — orfas de anotacao humana. A amostra efetiva fica, portanto,
# congelada por noteId; o algoritmo permanece como documentacao da selecao original.
QUAL_60_FROZEN = frozenset([
    '1639935770680078337', '1655712700171689984', '1689843429797175692', '1708038777610940549',
    '1760000020819103825', '1761231589785194785', '1761863207252029822', '1761891434389475646',
    '1762663085980958948', '1787884933765022194', '1798350866983891008', '1815075511498092968',
    '1823435417548374388', '1828475905272095265', '1828997016388518228', '1842682272262324483',
    '1856342402283225145', '1868118294474915987', '1870680545136439331', '1876223898854797725',
    '1877812049759416365', '1878062635578458242', '1878095415351922716', '1879837365738385874',
    '1881461849201521031', '1882118498148642865', '1884735731991519515', '1884778548310823225',
    '1885048614927425622', '1887258714580877671', '1889128334228885958', '1889176478891315215',
    '1916124309459808763', '1919473187269878071', '1936784166680502540', '1941004407832752217',
    '1991162547119014319', '1991189878722244743', '1993124451064406360', '1999916358998360139',
    '2007791917765943580', '2014522181053567129', '2019861320049271125', '2022468950043316327',
    '2023491294153052667', '2023720510257856645', '2025149321335673124', '2027161946576011568',
    '2028530269850910822', '2030386711545446739', '2031092784686387635', '2031096564928708748',
    '2032140619728965702', '2033327040481444237', '2034666298249023797', '2035165855806259553',
    '2035774913856840144', '2035937261900960010', '2036159146039595023', '2036235208928756034',
])
print(f'amostra congelada: {len(QUAL_60_FROZEN)} noteIds')

In [ ]:
N_QUAL_PER_CONSENSO = 20
CONSENSOS_QUAL = ['NMR', 'CRH', 'CRNH']

def _take(df, n, used, sort_cols, ascending):
    if n <= 0 or df.empty:
        return df.iloc[0:0]
    pool = df.loc[~df['noteId'].astype(str).isin(used)].copy()
    if pool.empty:
        return pool
    picked = pool.sort_values(sort_cols, ascending=ascending, kind='mergesort').head(n)
    used.update(picked['noteId'].astype(str).tolist())
    return picked

def _select_within(group, n):
    g = group.copy()
    if g.empty:
        return g
    g['span_gap'] = (g['n_e1'] - g['n_e2']).abs()
    both = g[(g['n_e1'] > 0) & (g['n_e2'] > 0)]
    used: set[str] = set()
    parts = []
    parts.append(_take(both, 6, used, ['f1_relaxed', 'kappa'], [True, True]))
    parts.append(_take(both, 4, used, ['span_gap', 'f1_relaxed'], [False, True]))
    mid  = both[both['f1_relaxed'].between(0.20, 0.55)]
    parts.append(_take(mid, 4, used, ['f1_relaxed', 'span_gap'], [True, False]))
    high = both[both['f1_relaxed'] >= 0.7]
    parts.append(_take(high, 3, used, ['f1_relaxed', 'kappa'], [False, False]))
    remaining = n - sum(len(p) for p in parts)
    if remaining > 0:
        div = g.loc[~g['noteId'].astype(str).isin(used)].copy()
        div['macrotheme_label'] = div['macrotheme_label'].fillna('Sem macrotema')
        div = (div.sort_values(['macrotheme_label', 'f1_relaxed'], ascending=[True, True])
                  .groupby('macrotheme_label', dropna=False).head(1))
        parts.append(_take(div, remaining, used, ['f1_relaxed', 'span_gap'], [True, False]))
    out = pd.concat(parts, ignore_index=False).drop_duplicates('noteId')
    if len(out) < n:
        filler = g.loc[~g['noteId'].astype(str).isin(out['noteId'].astype(str))]
        if not filler.empty:
            out = pd.concat([out, filler.sample(frac=1, random_state=RANDOM_SEED).head(n - len(out))])
    return out.head(n).reset_index(drop=True)

# === filtro ~is_meta aplicado ANTES do _select_within ===
pool = agreement[~agreement['is_meta']].copy()
print(f'pool sem meta: {len(pool):,} de {len(agreement):,}')

picks = [_select_within(pool[pool['consenso'].eq(c)], N_QUAL_PER_CONSENSO)
         for c in CONSENSOS_QUAL]
qual_algo = pd.concat(picks, ignore_index=True)

# === usa a amostra CONGELADA (ver celula anterior) ===
algo_ids = set(qual_algo['noteId'].astype(str))
if algo_ids != set(QUAL_60_FROZEN):
    n_diff = len(algo_ids.symmetric_difference(QUAL_60_FROZEN)) // 2
    print(f'AVISO: o algoritmo divergiu da amostra congelada em {n_diff} nota(s); '
          f'mantendo a congelada (as 60 ja anotadas).')
qual = pool[pool['noteId'].astype(str).isin(QUAL_60_FROZEN)].copy()
# (is_meta/meta_reason ja vem de `agreement`; nao re-mesclar para evitar colisao de colunas)
qual = qual.merge(
    sweep[['noteId', 'text', 'tweet_text', 'e1_spans', 'e2_spans']],
    on='noteId', how='left',
)
qual.to_parquet(QUALITATIVE_PARQUET, index=False)
qual.to_excel(QUALITATIVE_XLSX, index=False)

print(f'\nsalvas {len(qual)} notas qualitativas (amostra congelada) em:')
print(f'  {QUALITATIVE_PARQUET}')
print(f'  {QUALITATIVE_XLSX}')
print('\n--- composição por consenso ---')
print(qual['consenso'].value_counts().to_string())
assert len(qual) == 60, 'amostra congelada incompleta — confira QUAL_60_FROZEN vs sweep'
assert qual['is_meta'].sum() == 0, 'amostra qualitativa contém meta-notas — bug'

## 16. Captura de `reasoning_content` para as 60 qualitativas

Liga `enable_thinking=True` no Qwen apenas para esta passagem (triplica latência sem ganho mensurável em precisão de span — uso só para o show_case qualitativo da apresentação). `try/finally` garante que a global volte ao valor anterior mesmo se a iteração quebrar no meio.

In [ ]:
RUN_QUAL_REASONING = True

if RUN_QUAL_REASONING:
    global ENABLE_THINKING
    old_thinking = ENABLE_THINKING
    ENABLE_THINKING = True
    try:
        if REASONING_JSONL.exists():
            REASONING_JSONL.unlink()
        with REASONING_JSONL.open('a', encoding='utf-8') as f:
            for _, row in tqdm(qual.iterrows(), total=len(qual), desc='reasoning'):
                note_id = str(row['noteId'])
                res = e2_extract(row['text'], capture_reasoning=True)
                f.write(json.dumps({
                    'noteId': note_id,
                    'tweetId': str(row.get('tweetId') or ''),
                    'reasoning': res.get('reasoning'),
                    'spans_recall': _spans_to_payload(res.get('spans', [])),
                    'error': res.get('error'),
                }, ensure_ascii=False) + '\n')
                f.flush()
        print(f'\nsalvo em {REASONING_JSONL}')
    finally:
        ENABLE_THINKING = old_thinking
else:
    print('RUN_QUAL_REASONING=False — pulando recaptura')

## 17. Tradução PT-BR dos reasoning (preservando os originais)

O Qwen com `enable_thinking=True` retorna o raciocínio em inglês. Para a apresentação, traduzimos via Google (sem chave, via `deep-translator` — `googletrans` está abandonado). Os campos originais são preservados; uma coluna `reasoning_pt` é adicionada. Reasoning longo (>4.500 chars) é quebrado em pedaços e remontado.

In [ ]:
!pip install -q deep-translator

In [ ]:
import time
from deep_translator import GoogleTranslator

GOOGLE_FREE_CHAR_LIMIT = 4500   # margem do limite ~5000 do endpoint free

def chunk_for_translate(text: str, limit: int = GOOGLE_FREE_CHAR_LIMIT) -> list[str]:
    if len(text) <= limit:
        return [text]
    out, buf = [], ''
    for part in text.split('\n'):
        if len(part) > limit:
            for s in part.split('. '):
                seg = s + ('. ' if not s.endswith('.') else '')
                if len(buf) + len(seg) > limit:
                    if buf: out.append(buf); buf = ''
                buf += seg
        else:
            seg = part + '\n'
            if len(buf) + len(seg) > limit:
                if buf: out.append(buf); buf = ''
            buf += seg
    if buf: out.append(buf)
    return out

def translate_long(text: str, src='en', dst='pt') -> str:
    if not text:
        return ''
    tr = GoogleTranslator(source=src, target=dst)
    out = []
    for chunk in chunk_for_translate(text):
        for attempt in range(3):
            try:
                out.append(tr.translate(chunk))
                break
            except Exception:
                if attempt == 2:
                    raise
                time.sleep(2 ** attempt)
    return '\n'.join(out)

# --- carrega reasoning JSONL ---
records = []
with REASONING_JSONL.open(encoding='utf-8') as f:
    for ln in f:
        ln = ln.strip()
        if ln:
            records.append(json.loads(ln))
print(f'reasoning carregados: {len(records)}')

# --- traduz preservando original ---
for r in tqdm(records, desc='traduzindo'):
    if r.get('reasoning') and not r.get('reasoning_pt'):
        try:
            r['reasoning_pt'] = translate_long(r['reasoning'])
        except Exception as e:
            r['reasoning_pt'] = None
            r['reasoning_pt_error'] = f'{type(e).__name__}: {e}'

# --- regrava JSONL com reasoning_pt ao lado do original ---
with REASONING_JSONL.open('w', encoding='utf-8') as f:
    for r in records:
        f.write(json.dumps(r, ensure_ascii=False) + '\n')

# --- versão parquet pra plugar no relatório ---
REASONING_TRANSL_PARQUET = RUNS_DIR / 'reasoning_traduzido.parquet'
df_reason = pd.DataFrame(records)
df_reason['noteId'] = df_reason['noteId'].astype(str)
df_reason.to_parquet(REASONING_TRANSL_PARQUET, index=False)
print(f'\nsalvo: {REASONING_TRANSL_PARQUET}  '
      f'({int(df_reason["reasoning_pt"].notna().sum())}/{len(df_reason)} traduzidos)')

## 18. Dataset consolidado + empacotamento final

`dataset_anotado_final.parquet` reúne tudo o que produzimos: spans de E1 e E2, métricas por linha, flag de meta, status de seleção para anotação humana, modelo usado, latências. Todos os IDs como string. Pronto para ser carregado pelo app de anotação (`anotacao.html` / pasta `deploy/`).

In [ ]:
DATASET_PATH = DATA_DIR / 'dataset_anotado_final.parquet'

dataset = sweep.merge(
    agreement[['noteId', 'f1_strict', 'f1_relaxed', 'kappa']],
    on='noteId', how='left',
)
dataset['anotacao_humana_spans']    = None
dataset['anotacao_humana_anotador'] = None
dataset['anotacao_humana_status']   = 'nao_anotada'
dataset.loc[dataset['noteId'].isin(qual['noteId']),
            'anotacao_humana_status'] = 'selecionada_para_anotacao'

cols_finais = [
    'noteId', 'tweetId', 'consenso', 'macrotheme_label',
    'text', 'tweet_text', 'is_meta', 'meta_reason',
    'e1_spans', 'e2_spans', 'e2_align_levels', 'e2_err',
    'f1_strict', 'f1_relaxed', 'kappa',
    'e1_ms', 'e2_ms', 'model',
    'anotacao_humana_spans', 'anotacao_humana_anotador', 'anotacao_humana_status',
]
dataset = dataset[[c for c in cols_finais if c in dataset.columns]]
dataset['noteId']  = dataset['noteId'].astype(str)
dataset['tweetId'] = dataset['tweetId'].astype(str)
dataset.to_parquet(DATASET_PATH, index=False)
print(f'dataset_anotado_final: {dataset.shape}  '
      f'({DATASET_PATH.stat().st_size/1024:.1f} KB)')

### 18.1 Relatório operacional final

In [ ]:
from collections import Counter

n = len(sweep)
err = sweep['e2_err'].notna().sum()
print(f'[operacional] notas: {n:,} | erros E2: {err} ({100*err/n:.2f}%)')
print(f'  latência E2 mediana: {sweep["e2_ms"].median():.0f} ms  '
      f'| p95: {sweep["e2_ms"].quantile(.95):.0f} ms  '
      f'| total: {sweep["e2_ms"].sum()/1000/3600:.2f} h')

align_total = Counter()
for d in sweep['e2_align_levels']:
    if isinstance(d, dict):
        for k, v in d.items():
            align_total[k] += int(v)
tot = sum(align_total.values()) or 1
print(f'\n[alinhamento snippet→offset — {tot:,} snippets]')
for k in ['exact', 'normalized', 'unicode_normalized', 'regex', 'failed']:
    v = align_total.get(k, 0)
    flag = '  <- AÇÃO' if (k == 'failed' and v/tot > 0.03) else ''
    print(f'  {k:20s} {v:6d} ({100*v/tot:5.2f}%){flag}')

### 18.2 Zip de entrega

In [ ]:
import zipfile

ZIP_PATH = Path('entrega_v2.zip')
to_pack = [
    SUBSET_PATH, SWEEP_JSONL, AGREEMENT_PARQUET, TWEET_ALIGN_PARQUET,
    QUALITATIVE_PARQUET, QUALITATIVE_XLSX, REASONING_JSONL,
    REASONING_TRANSL_PARQUET, DATASET_PATH,
]
with zipfile.ZipFile(ZIP_PATH, 'w', zipfile.ZIP_DEFLATED) as zf:
    for p in to_pack:
        if Path(p).exists():
            zf.write(p, arcname=Path(p).name)
            print(f'  + {Path(p).name}  ({Path(p).stat().st_size/1024:.1f} KB)')
print(f'\nzip pronto: {ZIP_PATH}  ({ZIP_PATH.stat().st_size/1024/1024:.2f} MB)')

try:
    from google.colab import files
    files.download(str(ZIP_PATH))
except ImportError:
    print('(fora do Colab — baixe manualmente)')

## 19. Re-derivação do E1 (v2.1) — correção da ingestão de FONTE, preservando o E2

A execução original (v2) usava os offsets `inicio`/`fim` da tabela de entidades
diretamente; a auditoria pública do dataset (`quality_audit/`) mostrou que eles
frequentemente não se referem ao texto da nota — parte do erro de fronteira do E1 em FONTE
era, portanto, **artefato de ingestão**, não limite do método. Esta seção recomputa
**apenas o E1** sobre o dataset consolidado, com o loader corrigido da §5, **preservando
os spans do E2** da execução original (o LLM não é re-consultado). Também mede a latência
**fim-a-fim** do E1 (parse spaCy + regras, por nota, sem amortização) — na v2, `e1_ms`
cobria só a aplicação das regras.

**Pré-requisitos mínimos (nesta ordem):** §1 (config), §4 (schema/loader), §6 (spaCy),
§7 (regras E1) e §9 (métricas). **Não requer** §2–§3 (aquisição), §5, §8 (cliente
Qwen/API) nem o sweep — o dataset consolidado v2 é baixado do Drive automaticamente.

**Saídas:** `data/dataset_anotado_final_v3.parquet` (mesmo schema, com `e1_spans`
corrigidos, `e1_parse_ms`/`e1_total_ms` e `e1_version`) e um relatório antes/depois.

In [ ]:
# ====== 19. Re-derivacao E1 v2.1 (preserva E2; nao usa API) ======
import numpy as np
from collections import Counter

# Autossuficiencia: esta secao NAO requer §2-§3 (aquisicao), §5 (mapa sobre `subset`),
# §8 (API) nem §18. Define o que falta e baixa o dataset consolidado v2 se preciso.
if 'ENTITIES_PATH' not in globals():
    ENTITIES_PATH = DATA_DIR / ENTITIES_FILE
if 'HF_ENRICHED_REPO' not in globals():
    HF_ENRICHED_REPO = HF_REPO
V2_DATASET_PATH = DATA_DIR / 'dataset_anotado_final.parquet'   # saida da execucao v2
V3_DATASET_PATH = DATA_DIR / 'dataset_anotado_final_v3.parquet'
V2_DRIVE_FILE_ID = '1uOQa49x8XCPYrydKHCkP2XREA2CdP2Mj'         # mesmo arquivo lido pelo notebook 2
if not V2_DATASET_PATH.exists():
    import gdown
    gdown.download(f'https://drive.google.com/uc?id={V2_DRIVE_FILE_ID}',
                   str(V2_DATASET_PATH), quiet=False)

df_v2 = pd.read_parquet(V2_DATASET_PATH, engine='pyarrow')
df_v2['noteId'] = df_v2['noteId'].astype(str)
_n_e2 = df_v2['e2_spans'].apply(lambda p: len(p) if p is not None and hasattr(p, '__len__') else 0)
assert _n_e2.gt(0).sum() > 0, ('E2 vazio — leia com pyarrow/DuckDB '
                               '(fastparquet devolve as colunas aninhadas nulas).')
df_v2['_text'] = df_v2['text'].fillna('')

# Loader corrigido (§4/§5) sobre ESTAS notas — substitui o mapa global usado pelo e1_extract
ENTITY_SOURCE_SPANS = load_entity_source_spans(df_v2)

def _payload_spans(payload):
    return [Span(int(s['start']), int(s['end']), str(s['type']))
            for s in (payload or []) if s.get('start') is not None]

def _to_plain(payload):
    """Normaliza estruturas vindas do pyarrow para tipos python puros (regravacao segura)."""
    if payload is None or (isinstance(payload, float) and pd.isna(payload)):
        return None
    return [{'start': int(s['start']), 'end': int(s['end']), 'type': str(s['type'])}
            for s in payload]

recs = []
antes = Counter(); depois = Counter()
m_old = {'fs': [], 'fr': [], 'kp': []}
m_new = {'fs': [], 'fr': [], 'kp': []}
for _, row in tqdm(df_v2.iterrows(), total=len(df_v2), desc='E1 v2.1'):
    note_id = str(row['noteId']); text = row['_text']
    t0 = time.perf_counter()
    parse = parse_text(text)              # parse individual, cronometrado (sem lote)
    t1 = time.perf_counter()
    e1_new = e1_extract(text, note_id=note_id, parse=parse)
    t2 = time.perf_counter()

    e1_old = _payload_spans(row['e1_spans'])
    e2 = _payload_spans(row['e2_spans'])
    antes['spans'] += len(e1_old); antes['FONTE'] += sum(s.type == 'FONTE' for s in e1_old)
    depois['spans'] += len(e1_new); depois['FONTE'] += sum(s.type == 'FONTE' for s in e1_new)
    m_old['fs'].append(f1_strict(e1_old, e2)); m_new['fs'].append(f1_strict(e1_new, e2))
    m_old['fr'].append(f1_relaxed(e1_old, e2)); m_new['fr'].append(f1_relaxed(e1_new, e2))
    m_old['kp'].append(cohen_kappa_chars(text, e1_old, e2))
    m_new['kp'].append(cohen_kappa_chars(text, e1_new, e2))

    rec = {k: v for k, v in row.items() if k != '_text'}
    rec['e1_spans'] = [s.to_dict() for s in e1_new]
    rec['e2_spans'] = _to_plain(row['e2_spans'])
    rec['anotacao_humana_spans'] = _to_plain(row.get('anotacao_humana_spans'))
    if isinstance(row.get('e2_align_levels'), dict) or hasattr(row.get('e2_align_levels'), 'items'):
        rec['e2_align_levels'] = {str(k): int(v) for k, v in dict(row['e2_align_levels']).items()}
    rec['e1_ms'] = round((t2 - t1) * 1000, 2)          # so regras (comparavel ao v2)
    rec['e1_parse_ms'] = round((t1 - t0) * 1000, 2)    # parse spaCy individual
    rec['e1_total_ms'] = round((t2 - t0) * 1000, 2)    # fim-a-fim por nota
    rec['f1_strict'] = m_new['fs'][-1]
    rec['f1_relaxed'] = m_new['fr'][-1]
    rec['kappa'] = m_new['kp'][-1]
    rec['e1_version'] = 'v2.1_relocated_entities'
    recs.append(rec)

df_v3 = pd.DataFrame(recs)
df_v3.to_parquet(V3_DATASET_PATH, engine='pyarrow', index=False)

print('\n--- antes/depois (E1 x E2, medias por nota) ---')
print(f"v2  : {antes['spans']:,} spans E1 ({antes['FONTE']:,} FONTE) | "
      f"F1e={np.mean(m_old['fs']):.3f} F1r={np.mean(m_old['fr']):.3f} k={np.mean(m_old['kp']):.3f}")
print(f"v2.1: {depois['spans']:,} spans E1 ({depois['FONTE']:,} FONTE) | "
      f"F1e={np.mean(m_new['fs']):.3f} F1r={np.mean(m_new['fr']):.3f} k={np.mean(m_new['kp']):.3f}")
lat = df_v3[['e1_ms', 'e1_parse_ms', 'e1_total_ms']].median()
print(f"latencia E1 (mediana/nota): regras {lat['e1_ms']:.1f} ms | "
      f"parse {lat['e1_parse_ms']:.1f} ms | fim-a-fim {lat['e1_total_ms']:.1f} ms")
print(f'salvo: {V3_DATASET_PATH}')
# Proximo passo: substituir o conteudo do arquivo do Drive lido pelo notebook 2
# (DRIVE_FILE_ID la) por este v3, PRESERVANDO o file id.

---

**Encadeamento com o restante do projeto:**

1. `apps/anotador/` (estático; publicado em `anotador-argumentos.netlify.app`) apresenta o
   recorte de 60 notas para a anotação humana e exporta as marcações em JSON.
2. `notebook_conclusao.ipynb` (notebook 2) consome o dataset consolidado — após a §19, o
   `dataset_anotado_final_v3.parquet` — mais o(s) JSON(s) de anotador; faz a normalização
   BIO e calcula todas as medidas de avaliação.
3. `notebook_destilacao.ipynb` (notebook 3 / E3) treina modelos de sequência sobre o BIO
   (supervisão *silver* do E2, teste no *gold* humano).